In [ ]:
!pip install --force-reinstall --no-cache-dir transformers==4.37.2
!pip install sentence-transformers faiss-cpu

In [ ]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
from transformers import pipeline

In [ ]:
doc1 = """
Retrieval Augmented Generation (RAG) is a technique that combines
information retrieval with large language models. Instead of relying
only on the model's training data, the system retrieves relevant
documents from a knowledge base before generating a response.
"""

doc2 = """
A vector database stores vector embeddings of text. These embeddings
capture semantic meaning and allow similarity search. FAISS is a
popular vector database used for efficient nearest neighbor search.
"""

documents = [doc1, doc2]

In [ ]:
def split_text(text, chunk_size=150):
    chunks = []
    for i in range(0, len(text), chunk_size):
        chunks.append(text[i:i+chunk_size])
    return chunks

chunks = []
for doc in documents:
    chunks.extend(split_text(doc))

print(chunks)

In [ ]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

chunk_embeddings = embedding_model.encode(chunks)

print(chunk_embeddings.shape)

In [ ]:
dimension = chunk_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(np.array(chunk_embeddings))

print("Vectors stored:", index.ntotal)

In [ ]:
query = "What is Retrieval Augmented Generation?"

In [ ]:
query_embedding = embedding_model.encode([query])

k = 2
distances, indices = index.search(np.array(query_embedding), k)

retrieved_chunks = [chunks[i] for i in indices[0]]

print("Retrieved Context:\n")
for c in retrieved_chunks:
    print(c)

In [ ]:
context = " ".join(retrieved_chunks)

prompt = f"""
Use the context below to answer the question.

Context:
{context}

Question:
{query}

Answer:
"""

In [ ]:
generator = pipeline("text-generation", model="distilgpt2")

result = generator(prompt, max_length=150)

print(result[0]["generated_text"])